In [ ]:
# ============================================================
# Configuración (sin credenciales en el código)
# ============================================================
# Las credenciales se leen de un secret scope de Databricks (dbutils.secrets),
# cuyo valor Databricks redacta en la salida. Si el notebook se ejecuta fuera de
# Databricks, se leen de variables de entorno (.env). Nunca se escriben aquí.
import os

SECRET_SCOPE = "aiven_kafka"   # scope de Databricks (o respaldado por Azure Key Vault)


def _secret(key, env_var):
    """Lee del secret scope de Databricks; si no existe (local), del entorno."""
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=key)  # noqa: F821
    except Exception:
        val = os.getenv(env_var)
        if not val:
            raise ValueError(
                f"Falta el secreto '{key}'. Cárgalo en el scope '{SECRET_SCOPE}' "
                f"(Databricks) o define {env_var} en el entorno (.env)."
            )
        return val


# Credenciales sensibles -> secret scope / entorno (NUNCA hardcodear)
KAFKA_USER = _secret("kafka-user", "KAFKA_USER")
KAFKA_PASSWORD = _secret("kafka-password", "KAFKA_PASSWORD")

# Conexión no sensible -> configurable por entorno, con valor por defecto genérico
KAFKA_HOST = os.getenv("KAFKA_HOST", "kafka-<id>.aivencloud.com:<puerto>")
TOPIC = os.getenv("KAFKA_TOPIC", "reviews-nuevas")

CATALOGO = "proyecto_bda"
SCHEMA = "bda_schema"

# El certificado CA es público (no secreto): se monta en un Volume de Unity Catalog.
KAFKA_CA = os.getenv("KAFKA_CA", f"/Volumes/{CATALOGO}/{SCHEMA}/bda_volumen/ca.pem")

# Tabla Bronze de destino para el streaming
BRONZE_TABLE = f"{CATALOGO}.{SCHEMA}.bronze_reviews_stream"

# Tabla Bronze del histórico (capa batch del medallón). Se usa para el
# filtro anti-duplicados: una reseña que YA está en el histórico no se
# vuelve a insertar aunque vuelva a llegar por el stream.
BRONZE_BATCH_TABLE = f"{CATALOGO}.{SCHEMA}.bronze_reviews"

# Checkpoint: Spark Structured Streaming lo exige para sinks Delta.
CHECKPOINT_PATH = f"/Volumes/{CATALOGO}/{SCHEMA}/bda_volumen/_checkpoints/reviews_stream"

MAX_OFFSETS_PER_TRIGGER = 5000   # reseñas por micro-batch (back-pressure)
TRIGGER_INTERVAL = "10 seconds"  # frecuencia de micro-batches

## Gestión de credenciales con secretos de Databricks

Este notebook **no contiene credenciales en el código**, siguiendo las buenas prácticas de Databricks:

- Las credenciales de Kafka (usuario y contraseña de Aiven) se guardan en un **secret scope** y se leen con `dbutils.secrets.get(scope, key)`. Databricks **redacta** automáticamente el valor en logs y resultados (lo muestra como `[REDACTED]`), evitando fugas.
- En **Azure Databricks** se recomienda un scope **respaldado por Azure Key Vault**: las credenciales se administran y rotan en Key Vault sin tocar el notebook.
- El **certificado CA** (público, no secreto) se monta en un **Volume** de Unity Catalog.
- Fuera de Databricks (ejecución local), las mismas variables se leen del entorno (`.env`).

Crear el scope y cargar los secretos una sola vez con la Databricks CLI:

```bash
databricks secrets create-scope aiven_kafka
databricks secrets put-secret aiven_kafka kafka-user
databricks secrets put-secret aiven_kafka kafka-password
```

Alternativa recomendada en Azure (scope respaldado por Key Vault):

```bash
databricks secrets create-scope aiven_kafka \
  --scope-backend-type AZURE_KEYVAULT \
  --resource-id <resource-id-del-key-vault> \
  --dns-name https://<nombre-keyvault>.vault.azure.net/
```

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import from_json, col, to_timestamp, window
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,)

print(f"Destino: {BRONZE_TABLE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

# Verificar configuración
missing = [k for k, v in {
    "KAFKA_HOST": KAFKA_HOST,
    "KAFKA_USER": KAFKA_USER,
    "KAFKA_PASSWORD": KAFKA_PASSWORD,
}.items() if not v or "REEMPLAZA" in str(v)]
if missing:
    raise ValueError(f"Falta configurar: {', '.join(missing)}. Edita las variables en la celda 0.")

# Esquema de la reseña (Coincide con 1_producer.py)
esquema = StructType([
    StructField("place_id",       StringType()),
    StructField("id_review",      StringType()),
    StructField("caption",        StringType()),
    StructField("relative_date",  StringType()),
    StructField("review_date",    StringType()),
    StructField("retrieval_date", StringType()),
    StructField("rating",         DoubleType()),
    StructField("username",       StringType()),
    StructField("n_review_user",  DoubleType()),
    StructField("n_photo_user",   DoubleType()),
    StructField("url_user",       StringType()),
    StructField("url_source",     StringType()),
    StructField("ingest_ts",      StringType()),
])

jaas_config = (
    'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{KAFKA_USER}" password="{KAFKA_PASSWORD}";'
)

df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_HOST)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .option("maxOffsetsPerTrigger", MAX_OFFSETS_PER_TRIGGER)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("kafka.ssl.truststore.type", "PEM")
    .option("kafka.ssl.truststore.location", KAFKA_CA)
    .load()
)

df_parseado = (
    df_kafka
    .selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema).alias("d"))
    .select("d.*")
    .withColumn("rating", col("rating").cast("string"))
    .withColumn("n_review_user", col("n_review_user").cast("string"))
    .withColumn("n_photo_user", col("n_photo_user").cast("string"))
)

print("Streaming DataFrame creado. Esperando datos desde Kafka...")
print(f"Broker: {KAFKA_HOST}")
print(f"Topic: {TOPIC}")


## Filtro anti-duplicados contra el histórico (hash por registro)

Antes de escribir cada micro-lote en `bronze_reviews_stream` se verifica que las reseñas **no existan ya en el histórico**. El control usa un **hash por registro** (`record_hash = SHA-256` sobre `place_id, id_review, caption, rating, review_date, username`), de modo que se compara identidad de contenido, no solo el `id_review`:

1. **Dedup interno del micro-lote** — `dropDuplicates(["record_hash"])` elimina repeticiones dentro del mismo lote.
2. **Anti-join contra el histórico** — `left anti-join` por `record_hash` contra los hashes ya presentes en la capa batch (`bronze_reviews`) **y** en lo ya ingestado por el stream (`bronze_reviews_stream`). Solo lo verdaderamente nuevo se persiste.

Así se evita el doble conteo cuando una reseña ya capturada por el batch reaparece en el flujo, manteniendo la integridad de la arquitectura Lambda. El reproceso de Silver/Gold se dispara después de cada corrida del stream (job de Databricks).

In [ ]:
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit

# Campos que definen la identidad de una reseña para el hash de deduplicación.
# Si dos registros tienen el mismo hash, son el mismo dato aunque cambie el
# id_review o lleguen por rutas distintas (batch vs stream).
CAMPOS_HASH = ["place_id", "id_review", "caption", "rating", "review_date", "username"]


def con_hash(df):
    """Agrega la columna record_hash = SHA-256 de los campos de identidad."""
    partes = [coalesce(col(c).cast("string"), lit("")) for c in CAMPOS_HASH]
    return df.withColumn("record_hash", sha2(concat_ws("||", *partes), 256))


def hashes_historicos():
    """Conjunto de record_hash ya presentes en el histórico: capa batch
    (bronze_reviews) + lo ya ingestado por el stream (bronze_reviews_stream).
    Devuelve un DataFrame con una sola columna 'record_hash' o None si aún no
    existe ninguna de las dos tablas."""
    fuentes = []
    for t in (BRONZE_BATCH_TABLE, BRONZE_TABLE):
        if spark.catalog.tableExists(t):
            cols = spark.table(t).columns
            if all(c in cols for c in CAMPOS_HASH):
                fuentes.append(con_hash(spark.table(t)).select("record_hash"))
    if not fuentes:
        return None
    hist = fuentes[0]
    for extra in fuentes[1:]:
        hist = hist.unionByName(extra)
    return hist.dropDuplicates(["record_hash"])


def escribir_sin_duplicados(batch_df, batch_id):
    """Sink foreachBatch: agrega un hash por registro y descarta los que ya
    existen en el histórico antes de insertar en bronze_reviews_stream."""
    recibidas = batch_df.count()

    # 1) Hash por registro + dedup interno del micro-lote
    batch_df = con_hash(batch_df).dropDuplicates(["record_hash"])

    # 2) Anti-join por record_hash contra el histórico (batch + stream)
    hist = hashes_historicos()
    if hist is not None:
        nuevos = batch_df.join(hist, on="record_hash", how="left_anti")
    else:
        nuevos = batch_df  # primera corrida: aún no hay histórico

    n_nuevos = nuevos.count()
    descartadas = recibidas - n_nuevos
    print(f"[batch {batch_id}] recibidas={recibidas:,} | nuevas={n_nuevos:,} | duplicadas descartadas={descartadas:,}")

    if n_nuevos > 0:
        (nuevos.write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(BRONZE_TABLE))


query_bronze = (
    df_parseado.writeStream
    .foreachBatch(escribir_sin_duplicados)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)  # No deja processingTime 10 seg por el cluster
    .start()
)

query_bronze.awaitTermination()

print(f"Stream finalizado. Tabla destino: {BRONZE_TABLE}")
spark.sql(f"SELECT count(*) AS total_en_bronze_stream FROM {BRONZE_TABLE}").show()

In [ ]:
ultimos = spark.sql(f"""
  SELECT place_id, rating, caption, review_date, ingest_ts
  FROM {BRONZE_TABLE}
  ORDER BY ingest_ts DESC
  LIMIT 20
""")
print("Últimas reseñas recibidas del stream:")
display(ultimos)

python -c "
import pandas as pd
df = pd.read_parquet('dataset/reviews_dataset.parquet')
df.sample(frac=0.01, random_state=42).sort_values('review_date').to_csv('dataset/reviews_stream_sim.csv', index=False)
"

python kafka/1_producer.py --interval 0.2